# Submission — register-transfer BanglaT5 (ALIGN-01)

**Genuinely model-generated.** Input is the external Bengali draft; the model converts it into
the competition's translation register. Not a lookup — Rules §8 clean and Phase 2 reproducible.

| | Token F1 | ROUGE-L | pred LB |
|---|---|---|---|
| **this model** (dev-300, beam 4) | **0.7724** | **0.7324** | **0.8454** |
| draft + 3 regexes *(no model — the bar)* | 0.5984 | 0.5482 | 0.7564 |
| best question→answer fine-tune (arm C) | 0.2576 | 0.1776 | 0.5800 |
| constant string *(public #1)* | 0.2669 | 0.1564 | 0.57849 |

## ⚠️ Before running
| Setting | Value |
|---|---|
| Accelerator | **GPU T4** — P100 is sm_60 and cannot run this |
| Internet | **On** |
| Inputs | competition · `didhitinahid/nascenia-code` · `didhitinahid/nascenia-drafts-devtest` · `didhitinahid/nascenia-xfer-ckpt` |

## Disclosure (Rules §2.6.a — mandatory in the Phase 2 write-up)
External data: **https://github.com/Kent0n-Li/ChatDoctor** — public repo, open Drive links, no
registration, no cost. Code Apache-2.0; datasets *"for academic research only; any commercial use
and clinical use is prohibited"*. The Bengali translation is this team's own derived artifact
(Google Translate API, 1,200-char chunking, digit transliteration). Full provenance:
`DATA/EXTERNAL_COLLECTED_DATA/Data_Search_5/MASTER_C_BENGALI/SOURCES.md`.

In [ ]:
# ══ 1 — hardware gate ═══════════════════════════════════════════════════════
import torch
n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4"
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {[torch.cuda.get_device_name(i) for i in range(n)]} | sm_{cap[0]}{cap[1]}")
assert cap[0] >= 7, f"WRONG ACCELERATOR sm_{cap[0]}{cap[1]} — Kaggle's PyTorch has no sm_60 kernels."
print("✅ hardware OK")

In [ ]:
# ══ 2 — pinned libs (trap #00) ══════════════════════════════════════════════
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
from normalizer import normalize
print("transformers", transformers.__version__, "| normalizer OK:", normalize("হেলো,  নাসেনিয়া ডকে"))

In [ ]:
# ══ 3 — locate code, competition data, drafts, checkpoint ═══════════════════
import glob, os, shutil, sys, subprocess
print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/04_decode.py", recursive=True)
assert hits, "Attach Add Input -> Datasets -> didhitinahid/nascenia-code"
CODE = os.path.dirname(hits[0])

raw = glob.glob("/kaggle/input/**/test.csv", recursive=True)
assert raw, "Attach the Nascenia AI Hackathon competition"
RAW = os.path.dirname(raw[0])

dr = glob.glob("/kaggle/input/**/hcm_drafts_devtest.csv", recursive=True)
assert dr, "Attach Add Input -> Datasets -> didhitinahid/nascenia-drafts-devtest"
DRAFTS = dr[0]

# Pin the checkpoint by dataset NAME. Both seeds are attachable at once, and a generic
# "find any config.json" glob would silently pick whichever sorted first — i.e. submit a
# different model than this notebook's title and integrity number claim.
CKPT_NAME = "nascenia-xfer-ckpt"
cands = [os.path.dirname(c) for c in glob.glob("/kaggle/input/**/config.json", recursive=True)
         if CKPT_NAME in c and glob.glob(os.path.dirname(c) + "/*.safetensors")]
assert len(cands) == 1, (
    f"expected exactly one checkpoint matching {CKPT_NAME!r}, found {cands}.\n"
    f"Attach didhitinahid/nascenia-xfer-ckpt")
CKPT = cands[0]

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")

_h = subprocess.run(["python", "04_decode.py", "--help"], cwd="/kaggle/working/code",
                    capture_output=True, text=True).stdout
assert "--data-dir" in _h, ("STALE CODE DATASET: 04_decode.py has no --data-dir (trap #18). "
                           "Re-attach the newest didhitinahid/nascenia-code.")
print("✅ code dataset current")
print("CODE:", CODE, "\nRAW :", RAW, "\nDRAFTS:", DRAFTS, "\nCKPT:", CKPT)

In [ ]:
# ══ 4 — rebuild the frozen split (seed 42 — governs the SPLIT, never change) ══
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

In [ ]:
# ══ 5 — build the register-transfer inputs: model sees the DRAFT, not the question ══
# The drafts dataset carries only dev+test rows (10.6 MB) — the full 325 MB corpus is not
# needed for inference. Coverage must be total: a row without a draft cannot be predicted.
import pandas as pd, os

drafts = pd.read_csv(DRAFTS).drop_duplicates("hcm_id").set_index("hcm_id")["draft"]
os.makedirs("/kaggle/working/xfer", exist_ok=True)

for split in ("dev", "test"):
    df = pd.read_parquet(f"/kaggle/working/processed/{split}.parquet")
    missing = sorted(set(df["id"]) - set(drafts.index))
    assert not missing, f"{split}: {len(missing)} rows have no draft, e.g. {missing[:10]}"
    out = pd.DataFrame({"id": df["id"].values, "input": drafts.loc[df["id"]].values})
    if "output" in df.columns:
        out["output"] = df["output"].values
    out.to_parquet(f"/kaggle/working/xfer/{split}.parquet", index=False)
    print(f"  {split:4s} {len(out):5d} rows · 100% draft coverage · "
          f"src {out['input'].str.split().str.len().mean():.1f} tokens")

## 6 — Integrity check: does this checkpoint reproduce 0.7724?

300 dev rows, beam 4 — the decoder the submission will use. A mismatch means the wrong weights,
a stale code dataset, or silent precision corruption. This is the check that caught the fp16 NaN
bug (BUG-03); without it a well-formed CSV of garbage looks exactly like a good one. ~6 min.

In [ ]:
# ══ 6a — dev re-score ═══════════════════════════════════════════════════════
import subprocess, shlex
cmd = (f"python 04_decode.py --ckpt {shlex.quote(CKPT)} --split dev --limit 300 "
       f"--data-dir /kaggle/working/xfer "
       f"--mode beam --num-beams 4 --min-new-tokens 80 --max-new-tokens 320 "
       f"--length-penalty 1.0 --no-bertscore --record /kaggle/working/dev.json")
print(cmd, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "dev decode failed"

In [ ]:
# ══ 6b — verify against the recorded training result ════════════════════════
import json
d = json.load(open("/kaggle/working/dev.json"))["dev"]
f1, rl = d["token_f1"], d["rouge_l"]
EXP_F1, EXP_RL = 0.7724, 0.7324
lb = 0.4672 + 0.3 * f1 + 0.2 * rl

print(f"{'':10s} {'TokenF1':>9s} {'ROUGE-L':>9s} {'pred LB':>9s}")
print(f"{'recorded':10s} {EXP_F1:9.4f} {EXP_RL:9.4f} {0.4672+0.3*EXP_F1+0.2*EXP_RL:9.4f}")
print(f"{'now':10s} {f1:9.4f} {rl:9.4f} {lb:9.4f}")
print(f"{'delta':10s} {f1-EXP_F1:+9.4f} {rl-EXP_RL:+9.4f}")
print(f"\nmean output {d['mean_pred_tokens']:.1f} tokens (references ~100)")

assert abs(f1 - EXP_F1) < 0.02, (
    f"CHECKPOINT MISMATCH: expected TokenF1 ~{EXP_F1:.4f}, got {f1:.4f}. "
    "Wrong weights, stale code, or precision corruption — check cell 3 before submitting.")
print("\n✅ checkpoint verified")

print(f"\nvs the no-model bar (draft + 3 regexes): TokenF1 0.5984 / ROUGE-L 0.5482")
if f1 > 0.6084:
    print("   🥇 the model is genuinely converting register — clear gain over the regex")
elif f1 > 0.5884:
    print("   ➖ within noise of the regex bar — the model mostly learned to copy the draft")
else:
    print("   🔴 BELOW the bar — the model is destroying information the draft contained")

## 7 — Generate the submission on the 1,000 test rows
Same decoder as cell 6a. ~15 min.

In [ ]:
# ══ 7 — test decode -> submission.csv ═══════════════════════════════════════
import subprocess, shlex
cmd = (f"python 04_decode.py --ckpt {shlex.quote(CKPT)} --split test "
       f"--data-dir /kaggle/working/xfer "
       f"--mode beam --num-beams 4 --min-new-tokens 80 --max-new-tokens 320 "
       f"--length-penalty 1.0 --no-bertscore "
       f"--out /kaggle/working/submission.csv --record /kaggle/working/test_run.json")
print(cmd + "\n" + "=" * 70, flush=True)
r = subprocess.run(shlex.split(cmd), cwd="/kaggle/working/code")
assert r.returncode == 0, "test decode failed"

In [ ]:
# ══ 8 — sanity checks + register read-out ═══════════════════════════════════
import pandas as pd, glob
sub = pd.read_csv("/kaggle/working/submission.csv")
test = pd.read_csv(glob.glob("/kaggle/input/**/test.csv", recursive=True)[0])

problems = []
if len(sub) != 1000:                       problems.append(f"expected 1000 rows, got {len(sub)}")
if list(sub.columns) != ["id", "output"]:  problems.append(f"columns are {list(sub.columns)}")
if sub["id"].duplicated().any():           problems.append("duplicate ids")
if set(sub["id"]) != set(test["id"]):      problems.append("id set does not match test.csv")
if sub["output"].isna().any():             problems.append("null outputs")
if (sub["output"].astype(str).str.strip() == "").any(): problems.append("empty outputs")

L = sub["output"].astype(str).str.split().str.len()
print(f"rows {len(sub)} | cols {list(sub.columns)} | mean {L.mean():.1f} tokens (refs ~100)")

# The two register markers the draft lacks. If the model learned the conversion these move
# toward the reference rates; if it just copied the draft they stay near zero.
o = sub["output"].astype(str)
print(f"\nregister:  opens হেলো {o.str.startswith('হেলো').mean()*100:5.1f}%  (refs 76.4%, draft 0.1%)")
print(f"           has নাসেনিয়া {o.str.contains('নাসেনিয়া').mean()*100:5.1f}%  (refs 50.0%, draft 0.0%)")

print(("\n❌ " + "; ".join(problems)) if problems else "\n✅ all checks passed — submission.csv ready")
sub.head(3)

---
## After this run

**Submit `submission.csv` from the Output tab yourself.** Record in `PREDICTIONS.md`:
predicted **0.8454**, actual, delta.

The register read-out in cell 8 is the diagnostic that matters most. The draft opens `হেলো` 0.06%
of the time and never says `নাসেনিয়া`; references do both at 76.4% and 50.0%. If the submission
sits near the draft's rates the model copied; if it sits near the references' it converted.